# Magic: The Gathering rules — baseline RAG vs reranking

This notebook is the single code entry point. It starts with the official September 25, 2026 Comprehensive Rules PDF, extracts and validates numbered rules, creates tokenizer-safe chunks, then runs the two retrieval paths required by the assignment. It records ranks, answers, and latency without assuming that reranking will improve every question.

**Run order:** execute cells from top to bottom. Keep the PDF in the same directory as this notebook. At section 8, enter your Gemini API key only in the hidden password prompt. Do not paste the key into a notebook cell; the prompt input is not saved with the notebook.


In [1]:
%pip install -q pdfplumber sentence-transformers faiss-cpu pandas numpy google-genai



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import hashlib
import json
import re
import time
from urllib.request import urlretrieve
from collections import Counter
from pathlib import Path

import pdfplumber
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder

faiss.omp_set_num_threads(1)

ROOT = Path.cwd()
PDF_PATH = ROOT / "MagicCompRules_2026-09-25.pdf"
OUT = ROOT / "corpus"
OUT.mkdir(exist_ok=True)
SOURCE_URL = "https://media.wizards.com/2026/downloads/MagicCompRules%2020260925.pdf"
EXPECTED_SHA256 = "674241213075cd581dd3255567ebeec7b40ffae3a27c6a3bd99218af6b2b6659"
EMBEDDING_MODEL_ID = "BAAI/bge-base-en-v1.5"
RERANKER_MODEL_ID = "cross-encoder/ms-marco-MiniLM-L6-v2"
LLM_MODEL_ID = "gemini-3.5-flash-lite"
if not PDF_PATH.is_file():
    urlretrieve(SOURCE_URL, PDF_PATH)
actual_sha256 = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()
assert actual_sha256 == EXPECTED_SHA256, "The PDF differs from the verified source edition"
print("PDF SHA-256:", actual_sha256)

/opt/anaconda3/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


PDF SHA-256: 674241213075cd581dd3255567ebeec7b40ffae3a27c6a3bd99218af6b2b6659


## 1. Extract the original PDF

We retain PDF page numbers so every retrieved passage can be checked against the source. The contents begin on page 6, and the glossary begins on page 256. Introductory pages and credits are not useful retrieval targets.

In [3]:
with pdfplumber.open(PDF_PATH) as pdf:
    pages = [(number, page.extract_text() or "") for number, page in enumerate(pdf.pages, 1)]

assert len(pages) == 312
assert all(text.strip() for _, text in pages)
assert "September 25, 2026" in pages[0][1]
assert pages[255][1].startswith("Glossary")
assert pages[310][1].startswith("Credits")
print(f"{len(pages)} pages extracted; {sum(len(text) for _, text in pages):,} characters")
print(pages[5][1][:700])

312 pages extracted; 947,879 characters
1. Game Concepts
100. General
100.1. These Magic rules apply to any Magic game with two or more players, including two-player
games and multiplayer games.
100.1a A two-player game is a game that begins with only two players.
100.1b A multiplayer game is a game that begins with more than two players. See section 8,
“Multiplayer Rules.”
100.2. To play, each player needs their own deck of traditional Magic cards, small items to represent
any tokens and counters, and some way to clearly track life totals.
100.2a In constructed play (a way of playing in which each player creates their own deck ahead of
time), each deck has a minimum deck size of 60 cards. A constructed deck may contain any
number


## 2. Parse numbered rules

A rule or subrule remains an atomic unit where possible. Lines that continue onto the next page are attached to the same rule. Examples stay with the rule they explain.

In [4]:
RULE_START = re.compile(r"^(\d{3}\.\d+[a-z]?)(?:\.)?\s+([A-Z“‘(].+)$")
SECTION_START = re.compile(r"^(\d{3})\.\s+([A-Z][^\n]+)$")
units = []
current = None
section = ""

for page_no, page_text in pages[5:255]:
    for raw in page_text.splitlines():
        line = raw.strip()
        if not line:
            continue
        rule_match = RULE_START.match(line)
        section_match = SECTION_START.match(line)
        if section_match and not rule_match:
            section = line
            continue
        if rule_match:
            if current is not None:
                units.append(current)
            current = {
                "rule_id": rule_match.group(1),
                "section": section,
                "page_start": page_no,
                "page_end": page_no,
                "text": line,
            }
        elif current is not None:
            current["text"] += " " + line
            current["page_end"] = page_no
if current is not None:
    units.append(current)

rule_ids = [unit["rule_id"] for unit in units]
duplicates = [key for key, count in Counter(rule_ids).items() if count > 1]
assert not duplicates, f"Duplicate rule IDs: {duplicates[:10]}"
assert "101.2" in rule_ids and "903.5" in rule_ids
print(f"{len(units):,} unique numbered rules")
print(next(unit for unit in units if unit["rule_id"] == "101.2"))

3,163 unique numbered rules
{'rule_id': '101.2', 'section': '101. The Magic Golden Rules', 'page_start': 7, 'page_end': 7, 'text': '101.2. When a rule or effect allows or directs something to happen, and another effect states that it can’t happen, the “can’t” effect takes precedence. Example: If one effect reads “You may play an additional land this turn” and another reads “You can’t play lands this turn,” the effect that precludes you from playing lands wins.'}


## 3. Load the two retrieval models and inspect their input limits

The same chunks will be used by both workflows. The Cross-Encoder sees the query and chunk together, so we reserve space for the query when setting a chunk limit.

In [5]:
embedder = SentenceTransformer(EMBEDDING_MODEL_ID)
reranker = CrossEncoder(RERANKER_MODEL_ID)
embed_tokenizer = embedder.tokenizer
rerank_tokenizer = reranker.tokenizer
EMBED_MAX = min(int(embedder.max_seq_length), 512)
PAIR_MAX = min(int(rerank_tokenizer.model_max_length), 512)
QUERY_RESERVE = 80
CHUNK_TOKEN_LIMIT = min(EMBED_MAX - 2, PAIR_MAX - QUERY_RESERVE - 3, 420)
assert CHUNK_TOKEN_LIMIT > 100
print({"embedding_max": EMBED_MAX, "pair_max": PAIR_MAX, "chunk_token_limit": CHUNK_TOKEN_LIMIT})

{'embedding_max': 512, 'pair_max': 512, 'chunk_token_limit': 420}


## 4. Make rule-aware, tokenizer-safe chunks

Consecutive rules are grouped only within their section. We check both tokenizers. An unusually long individual rule is split into sequential parts, retaining its rule ID and page reference. Glossary passages are grouped within a page.

In [6]:
def token_cost(text: str) -> int:
    return max(
        len(embed_tokenizer(text, add_special_tokens=False, verbose=False)["input_ids"]),
        len(rerank_tokenizer(text, add_special_tokens=False, verbose=False)["input_ids"]),
    )

def fits(text: str) -> bool:
    return token_cost(text) <= CHUNK_TOKEN_LIMIT

def split_long_text(text: str, prefix: str) -> list[str]:
    pieces = []
    words = text.split()
    start = 0
    while start < len(words):
        low, high = start + 1, len(words)
        best = None
        while low <= high:
            mid = (low + high) // 2
            candidate = prefix + " " + " ".join(words[start:mid])
            if fits(candidate):
                best = mid
                low = mid + 1
            else:
                high = mid - 1
        if best is None:
            raise ValueError("One word exceeds the model limit")
        pieces.append(prefix + " " + " ".join(words[start:best]))
        start = best
    return pieces

atomic = []
for unit in units:
    header = unit["section"] + "\n"
    if fits(header + unit["text"]):
        atomic.append(unit)
    else:
        parts = split_long_text(unit["text"], header + unit["rule_id"] + " (continued):")
        for part_no, part in enumerate(parts, 1):
            atomic.append({**unit, "text": part.replace(header, "", 1), "part": part_no})

chunks = []
batch = []

def flush_rule_batch():
    if not batch:
        return
    chunks.append({
        "chunk_id": f"rule-{len(chunks) + 1:04d}",
        "kind": "rules",
        "section": batch[0]["section"],
        "rule_ids": list(dict.fromkeys(unit["rule_id"] for unit in batch)),
        "page_start": batch[0]["page_start"],
        "page_end": batch[-1]["page_end"],
        "text": batch[0]["section"] + "\n" + "\n".join(unit["text"] for unit in batch),
    })
    batch.clear()

for unit in atomic:
    if batch:
        candidate = batch[0]["section"] + "\n" + "\n".join(
            item["text"] for item in [*batch, unit]
        )
        if unit["section"] != batch[-1]["section"] or not fits(candidate):
            flush_rule_batch()
    batch.append(unit)
flush_rule_batch()

for page_no, page_text in pages[255:310]:
    lines = page_text.splitlines()
    if lines and lines[0].strip() == "Glossary":
        lines = lines[1:]
    passage = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        candidate = "Glossary\n" + "\n".join([*passage, line])
        if passage and not fits(candidate):
            chunks.append({
                "chunk_id": f"glossary-{len(chunks) + 1:04d}",
                "kind": "glossary",
                "section": "Glossary",
                "rule_ids": [],
                "page_start": page_no,
                "page_end": page_no,
                "text": "Glossary\n" + "\n".join(passage),
            })
            passage = []
        if not fits("Glossary\n" + line):
            parts = split_long_text(line, "Glossary")
            for part in parts:
                chunks.append({
                    "chunk_id": f"glossary-{len(chunks) + 1:04d}",
                    "kind": "glossary",
                    "section": "Glossary",
                    "rule_ids": [],
                    "page_start": page_no,
                    "page_end": page_no,
                    "text": part,
                })
        else:
            passage.append(line)
    if passage:
        chunks.append({
            "chunk_id": f"glossary-{len(chunks) + 1:04d}",
            "kind": "glossary",
            "section": "Glossary",
            "rule_ids": [],
            "page_start": page_no,
            "page_end": page_no,
            "text": "Glossary\n" + "\n".join(passage),
        })

assert all(fits(chunk["text"]) for chunk in chunks)
assert {unit["rule_id"] for unit in units} == {
    rule_id for chunk in chunks for rule_id in chunk["rule_ids"]
}
print(f"{len(chunks)} chunks:", Counter(chunk["kind"] for chunk in chunks))
print("Longest chunk in model tokens:", max(token_cost(chunk["text"]) for chunk in chunks))

707 chunks: Counter({'rules': 598, 'glossary': 109})
Longest chunk in model tokens: 420


## 5. Save and inspect the corpus

The JSONL file is generated by this notebook. Source edition, checksum, model choices, and chunking method are written to a manifest for reproducibility.

In [7]:
for chunk in chunks:
    assert 6 <= chunk["page_start"] <= chunk["page_end"] <= 310
    assert chunk["text"].strip()
    chunk["token_count"] = token_cost(chunk["text"])

with (OUT / "chunks.jsonl").open("w", encoding="utf-8") as file:
    for chunk in chunks:
        file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

manifest = {
    "title": "Magic: The Gathering Comprehensive Rules",
    "edition": "2026-09-25",
    "source_url": SOURCE_URL,
    "source_pdf": PDF_PATH.name,
    "source_sha256": hashlib.sha256(PDF_PATH.read_bytes()).hexdigest(),
    "pdf_pages": len(pages),
    "numbered_rule_units": len(units),
    "chunks": len(chunks),
    "embedding_model": EMBEDDING_MODEL_ID,
    "reranker_model": RERANKER_MODEL_ID,
    "answer_model": LLM_MODEL_ID,
    "embedding_max_tokens": EMBED_MAX,
    "pair_max_tokens": PAIR_MAX,
    "chunk_token_limit": CHUNK_TOKEN_LIMIT,
    "note": "Rules and glossary included; introduction, contents, and credits excluded.",
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
display(pd.DataFrame(chunks)[["chunk_id", "kind", "section", "rule_ids", "page_start", "page_end", "token_count"]].head(8))
print("Corpus saved to", OUT)

,chunk_id,kind,section,rule_ids,page_start,page_end,token_count
0,rule-0001,rules,100. General,"[100.1, 100.1a, 100.1b, 100.2, 100.2a, 100.2b,...",6,6,409
1,rule-0002,rules,100. General,"[100.4, 100.4a, 100.4b, 100.4c, 100.4d, 100.5,...",6,7,396
2,rule-0003,rules,100. General,"[100.6b, 100.7]",7,7,107
3,rule-0004,rules,101. The Magic Golden Rules,"[101.1, 101.2, 101.2a, 101.3, 101.4]",7,7,370
4,rule-0005,rules,101. The Magic Golden Rules,"[101.4a, 101.4b, 101.4c, 101.4d, 101.4e]",7,8,210
5,rule-0006,rules,102. Players,"[102.1, 102.2, 102.3, 102.4]",8,8,151
6,rule-0007,rules,103. Starting the Game,"[103.1, 103.1a, 103.1b, 103.1c, 103.2, 103.2a,...",8,8,408
7,rule-0008,rules,103. Starting the Game,"[103.2c, 103.2d, 103.2e, 103.3, 103.3a, 103.4,...",8,9,410


Corpus saved to /Users/tanishaupadhyay/Documents/Magic_The_RAG/corpus


## 6. Build the dense vector index

The Bi-Encoder embeds every chunk once. FAISS stores normalized vectors and returns the top 10 by inner product, equivalent to cosine similarity for normalized vectors.

In [8]:
index_start = time.perf_counter()
passage_vectors = embedder.encode(
    [chunk["text"] for chunk in chunks],
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")
index = faiss.IndexFlatIP(passage_vectors.shape[1])
index.add(passage_vectors)
index_seconds = time.perf_counter() - index_start
assert index.ntotal == len(chunks)
print(f"Indexed {index.ntotal} passages in {index_seconds:.2f}s")

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Indexed 707 passages in 27.75s


## 7. Run both retrieval paths

Baseline: take the first 3 of the Bi-Encoder's top 10. Advanced: score those **same 10** candidates with the Cross-Encoder, then take its first 3. Times below exclude one-time model loading and index construction.

In [9]:
def retrieve(query: str) -> dict:
    start = time.perf_counter()
    query_vector = embedder.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    scores, positions = index.search(query_vector, 10)
    baseline_seconds = time.perf_counter() - start
    candidates = [
        {**chunks[int(position)], "bi_rank": rank, "bi_score": float(score)}
        for rank, (position, score) in enumerate(zip(positions[0], scores[0]), 1)
        if position >= 0
    ]
    assert len(candidates) == 10

    pair_lengths = [len(rerank_tokenizer(query, item["text"], truncation=False)["input_ids"]) for item in candidates]
    assert max(pair_lengths) <= PAIR_MAX, f"Query and passage exceed Cross-Encoder limit: {max(pair_lengths)}"

    start = time.perf_counter()
    ce_scores = reranker.predict(
        [(query, item["text"]) for item in candidates],
        show_progress_bar=False,
    )
    rerank_seconds = time.perf_counter() - start
    for item, score in zip(candidates, ce_scores):
        item["ce_score"] = float(score)
    reranked = sorted(candidates, key=lambda item: item["ce_score"], reverse=True)
    for rank, item in enumerate(reranked, 1):
        item["ce_rank"] = rank

    return {
        "query": query,
        "candidates": candidates,
        "baseline_top3": candidates[:3],
        "reranked_top3": reranked[:3],
        "baseline_seconds": baseline_seconds,
        "rerank_seconds": rerank_seconds,
        "advanced_seconds": baseline_seconds + rerank_seconds,
    }

pilot = retrieve("If one effect lets me play an additional land but another says I cannot play lands, can I play it?")
display(pd.DataFrame(pilot["candidates"])[
    ["chunk_id", "rule_ids", "bi_rank", "bi_score", "ce_rank", "ce_score"]
].sort_values("bi_rank"))
print({key: pilot[key] for key in ["baseline_seconds", "rerank_seconds", "advanced_seconds"]})

,chunk_id,rule_ids,bi_rank,bi_score,ce_rank,ce_score
0,rule-0151,"[305.1, 305.2, 305.2a, 305.2b, 305.3, 305.4, 3...",1,0.755349,2,3.390915
1,rule-0320,"[701.18a, 701.18b, 701.18c, 701.18d, 701.18e, ...",2,0.733757,3,2.810822
2,rule-0268,"[609.4a, 609.4b, 609.5, 609.6, 609.7]",3,0.701827,4,-0.257580
3,rule-0192,"[505.5, 505.6, 505.6a, 505.6b]",4,0.700095,5,-1.246058
4,rule-0184,"[500.7, 500.8, 500.9, 500.10, 500.10a, 500.11]",5,0.699320,8,-5.886896
5,rule-0267,"[609.1, 609.2, 609.3, 609.4]",6,0.698784,6,-2.713998
6,rule-0498,"[712.21d, 712.21e]",7,0.697740,9,-7.857586
7,rule-0004,"[101.1, 101.2, 101.2a, 101.3, 101.4]",8,0.695854,1,4.106308
8,rule-0152,"[305.6, 305.7, 305.8, 305.9]",9,0.693209,7,-4.102918
9,rule-0033,"[108.3b, 108.4, 108.4a, 108.5, 108.6]",10,0.686586,10,-10.208451


{'baseline_seconds': 0.07072087499545887, 'rerank_seconds': 0.24485012504737824, 'advanced_seconds': 0.3155710000428371}


## 8. Generate answers from the top 3 passages

Both paths use `gemini-3.5-flash-lite` through the Gemini API with identical prompts and generation settings. Enter the API key in the hidden prompt below when running the notebook. It stays in kernel memory and is not saved in this file or its outputs. Gemini API usage follows your Google project's pricing and rate limits. Check all generated answers against the cited rules.


In [10]:
from getpass import getpass
from google import genai
from google.genai import types

# getpass input is hidden and is never written to a code cell or notebook output.
gemini_api_key = getpass("Gemini API key (input hidden): ").strip()
if not gemini_api_key:
    raise ValueError("Enter a Gemini API key in the hidden prompt to continue.")
gemini_client = genai.Client(api_key=gemini_api_key)
del gemini_api_key
GEMINI_CONFIG = types.GenerateContentConfig(max_output_tokens=512)
print(f"Gemini client configured for {LLM_MODEL_ID}; key not displayed or saved.")


Gemini API key (input hidden):  ········


Gemini client configured for gemini-3.5-flash-lite; key not displayed or saved.


In [11]:
PROMPT_TEMPLATE = """Answer the Magic: The Gathering rules question using only the passages below.
If they do not establish the answer, say that the supplied passages are insufficient.
Be concise and cite rule numbers or PDF pages present in the passages.

Question: {query}

Passages:
{passages}
"""

def make_prompt(query: str, selected: list[dict]) -> str:
    passages = "\n\n".join(
        f"[{item['chunk_id']}, PDF p. {item['page_start']}"
        f"{'-' + str(item['page_end']) if item['page_end'] != item['page_start'] else ''}]\n"
        f"{item['text']}"
        for item in selected
    )
    return PROMPT_TEMPLATE.format(query=query, passages=passages)

def generate_answer(query: str, selected: list[dict]) -> tuple[str, float]:
    prompt = make_prompt(query, selected)
    start = time.perf_counter()
    response = gemini_client.models.generate_content(
        model=LLM_MODEL_ID,
        contents=prompt,
        config=GEMINI_CONFIG,
    )
    elapsed = time.perf_counter() - start
    answer = (response.text or "").strip()
    if not answer:
        raise RuntimeError("Gemini returned no text; inspect the API response before continuing.")
    return answer, elapsed

answer_1, llm_time_1 = generate_answer(pilot["query"], pilot["baseline_top3"])
answer_2, llm_time_2 = generate_answer(pilot["query"], pilot["reranked_top3"])
print("Baseline answer:", answer_1)
print("Reranked answer:", answer_2)
print("LLM seconds:", llm_time_1, llm_time_2)


Baseline answer: The supplied passages are insufficient.
Reranked answer: No, you cannot play it. When a rule or effect allows something and another states it can't happen, the "can't" effect takes precedence (Rule 101.2, PDF p. 7).
LLM seconds: 2.1390292499563657 1.3459918750450015


## 9. Evaluate rank shifts and latency

These are **starter questions**, not claimed examples of reranking success. Check each expected supporting rule against the PDF before treating it as a gold label. A query with several gold rules counts as a hit only when **all** required rules are in the top 3. Add more independently verified questions, including failures and regressions. The assignment needs at least two observed cases where reranking restores the required evidence and improves the answer.

In [12]:
questions = [
    {
        "query": "If I may play an additional land but another effect says I cannot play lands, can I play one?",
        "gold_rule_ids": ["101.2"],
    },
    {
        "query": "When card text directly contradicts a general game rule, which takes precedence?",
        "gold_rule_ids": ["101.1"],
    },
    {
        "query": "What is the difference between an activated ability and a triggered ability?",
        "gold_rule_ids": ["602.1", "603.1"],
    },
    {
        "query": "How do replacement effects differ from prevention effects?",
        "gold_rule_ids": ["614.1", "615.1"],
    },
]

def gold_ranks(items: list[dict], gold_rule_ids: list[str], rank_key: str) -> dict[str, int | None]:
    return {
        rule_id: min(
            (item[rank_key] for item in items if rule_id in item["rule_ids"]),
            default=None,
        )
        for rule_id in gold_rule_ids
    }

def all_in_top_k(ranks: dict[str, int | None], k: int) -> bool:
    return all(rank is not None and rank <= k for rank in ranks.values())

results = []
for item in questions:
    run = retrieve(item["query"])
    first_answer, first_llm_seconds = generate_answer(item["query"], run["baseline_top3"])
    second_answer, second_llm_seconds = generate_answer(item["query"], run["reranked_top3"])
    gold = item["gold_rule_ids"]
    bi_ranks = gold_ranks(run["candidates"], gold, "bi_rank")
    ce_ranks = gold_ranks(run["candidates"], gold, "ce_rank")
    gold_in_top10 = all_in_top_k(bi_ranks, 10)
    baseline_hit_at_3 = all_in_top_k(bi_ranks, 3)
    reranked_hit_at_3 = all_in_top_k(ce_ranks, 3)
    bi_rank = max(bi_ranks.values()) if gold_in_top10 else None
    ce_rank = max(ce_ranks.values()) if gold_in_top10 else None
    results.append({
        "query": item["query"],
        "gold_rule_ids": gold,
        "bi_rank": bi_rank,
        "ce_rank": ce_rank,
        "bi_rule_ranks": bi_ranks,
        "ce_rule_ranks": ce_ranks,
        "gold_in_top10": gold_in_top10,
        "baseline_hit_at_3": baseline_hit_at_3,
        "reranked_hit_at_3": reranked_hit_at_3,
        "recovered": gold_in_top10 and not baseline_hit_at_3 and reranked_hit_at_3,
        "baseline_top3": [x["chunk_id"] for x in run["baseline_top3"]],
        "reranked_top3": [x["chunk_id"] for x in run["reranked_top3"]],
        "baseline_seconds": run["baseline_seconds"],
        "rerank_seconds": run["rerank_seconds"],
        "advanced_seconds": run["advanced_seconds"],
        "answer_1": first_answer,
        "answer_2": second_answer,
        "llm_seconds_1": first_llm_seconds,
        "llm_seconds_2": second_llm_seconds,
    })

summary = pd.DataFrame(results)
display(summary[["query", "gold_rule_ids", "bi_rank", "ce_rank", "recovered",
                 "baseline_seconds", "rerank_seconds", "advanced_seconds"]])
print("Recall@3 baseline:", summary["baseline_hit_at_3"].mean())
print("Recall@3 reranked:", summary["reranked_hit_at_3"].mean())
print("Recovered cases:", int(summary["recovered"].sum()))
print("Mean retrieval seconds:", summary[["baseline_seconds", "rerank_seconds", "advanced_seconds"]].mean().to_dict())
(OUT / "evaluation_results.json").write_text(
    json.dumps(results, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)

,query,gold_rule_ids,bi_rank,ce_rank,recovered,baseline_seconds,rerank_seconds,advanced_seconds
0,If I may play an additional land but another e...,[101.2],5.0,2.0,True,0.083615,0.154486,0.238101
1,When card text directly contradicts a general ...,[101.1],1.0,1.0,False,0.110323,0.149973,0.260296
2,What is the difference between an activated ab...,"[602.1, 603.1]",NaN,NaN,False,0.109442,0.135222,0.244664
3,How do replacement effects differ from prevent...,"[614.1, 615.1]",6.0,3.0,True,0.107042,0.145213,0.252255


Recall@3 baseline: 0.25
Recall@3 reranked: 0.75
Recovered cases: 2
Mean retrieval seconds: {'baseline_seconds': 0.10260570851096418, 'rerank_seconds': 0.14622339599009138, 'advanced_seconds': 0.24882910450105555}


5780

## 10. Inspect actual failure cases

Only cases that satisfy the assignment's rank condition appear below. Review the full passages and both answers before writing the report in your own words. If fewer than two appear, expand the question set or revise chunking and rerun; do not invent rank shifts.

In [13]:
for row in results:
    if row["recovered"]:
        print("=" * 80)
        print("QUESTION:", row["query"])
        print("GOLD RULES:", row["gold_rule_ids"])
        print("RANK SHIFT:", row["bi_rank"], "->", row["ce_rank"])
        print("BASELINE TOP 3:", row["baseline_top3"])
        print("RERANKED TOP 3:", row["reranked_top3"])
        print("ANSWER 1:", row["answer_1"])
        print("ANSWER 2:", row["answer_2"])
if not any(row["recovered"] for row in results):
    print("No qualifying recovery in this starter set. Add independently verified stress questions.")

QUESTION: If I may play an additional land but another effect says I cannot play lands, can I play one?
GOLD RULES: ['101.2']
RANK SHIFT: 5 -> 2
BASELINE TOP 3: ['rule-0151', 'rule-0320', 'rule-0192']
RERANKED TOP 3: ['rule-0151', 'rule-0004', 'rule-0320']
ANSWER 1: The supplied passages are insufficient.
ANSWER 2: No, you cannot play one. When a rule or effect allows or directs something to happen, and another effect states that it can’t happen, the “can’t” effect takes precedence (rule 101.2).
QUESTION: How do replacement effects differ from prevention effects?
GOLD RULES: ['614.1', '615.1']
RANK SHIFT: 6 -> 3
BASELINE TOP 3: ['rule-0303', 'glossary-0674', 'rule-0288']
RERANKED TOP 3: ['rule-0303', 'rule-0288', 'rule-0297']
ANSWER 1: Based on the supplied passages, a prevention effect is a kind of continuous effect that watches for a damage event and completely or partially prevents the damage that would be dealt (Glossary, PDF p. 293). In contrast, a replacement effect watches for a

## 11. Audit answer correctness separately from retrieval

The rank-shift metric checks whether relevant rule chunks reached the top 3. It does **not** prove that Gemini used those rules correctly. The worksheet below leaves correctness judgments blank for manual review against the saved answers and original PDF. This cell reads the completed evaluation results and does not call the model again.


In [14]:
# Review each new Gemini answer against the retrieved passages and original PDF.
# Fill in the judgments and notes yourself before using them in the report.
answer_audit = [
    {
        "query": run["query"],
        "gold_rule_ids": run["gold_rule_ids"],
        "bi_rank": run["bi_rank"],
        "ce_rank": run["ce_rank"],
        "recovered": run["recovered"],
        "baseline_judgment": "To review",
        "baseline_note": "",
        "reranked_judgment": "To review",
        "reranked_note": "",
    }
    for run in results
]
display(pd.DataFrame(answer_audit)[[
    "query", "bi_rank", "ce_rank", "recovered",
    "baseline_judgment", "reranked_judgment",
]])
(OUT / "answer_audit.json").write_text(
    json.dumps(answer_audit, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print("Review worksheet saved to", OUT / "answer_audit.json")
print("Check both answers against the original rule text before writing your report.")


,query,bi_rank,ce_rank,recovered,baseline_judgment,reranked_judgment
0,If I may play an additional land but another e...,5.0,2.0,True,To review,To review
1,When card text directly contradicts a general ...,1.0,1.0,False,To review,To review
2,What is the difference between an activated ab...,NaN,NaN,False,To review,To review
3,How do replacement effects differ from prevent...,6.0,3.0,True,To review,To review


Review worksheet saved to /Users/tanishaupadhyay/Documents/Magic_The_RAG/corpus/answer_audit.json
Check both answers against the original rule text before writing your report.
